# 📜 Notebook 3: Event Sourcing

Using events to build reliable workflows.

## Learning Objectives

By the end of this notebook, you'll understand:
- Event sourcing basics
- How events enable recovery
- Worker-based processing
- Why this is still complex

In [ ]:
import redis
import json
import uuid
import time
import random
from datetime import datetime

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.flushall()

print("✅ Redis connected!")

## 📜 What is Event Sourcing?

In [ ]:
print("📜 Event Sourcing Explained")
print("=" * 60)
print("""
TRADITIONAL: Store current state
─────────────────────────────────────────────────────────────
  Order #42: status='shipped', total=$99.99
  
  Lost history! How did we get here?

EVENT SOURCING: Store sequence of events
─────────────────────────────────────────────────────────────
  Event 1: OrderCreated { order_id: 42, items: [...] }
  Event 2: PaymentCharged { order_id: 42, amount: 99.99 }
  Event 3: InventoryReserved { order_id: 42, sku: 'ABC' }
  Event 4: OrderShipped { order_id: 42, tracking: '1Z...' }
  
  Full history! Can replay to rebuild state.

FOR WORKFLOWS:
─────────────────────────────────────────────────────────────
  Events trigger next steps automatically!
  
  [OrderCreated] ──► Payment Worker ──► [PaymentCharged]
                                              │
                                              ▼
  [InventoryReserved] ◄── Inventory Worker ◄──┘
          │
          ▼
  Shipping Worker ──► [OrderShipped]
""")

In [ ]:
class EventStore:
    def __init__(self, redis_client):
        self.redis = redis_client
        self.stream = "events"
    
    def append(self, event_type: str, data: dict) -> str:
        event = {
            "type": event_type,
            "data": json.dumps(data),
            "timestamp": datetime.now().isoformat()
        }
        event_id = self.redis.xadd(self.stream, event)
        return event_id
    
    def read_all(self) -> list:
        events = self.redis.xrange(self.stream, "-", "+")
        return [
            {
                "id": e[0],
                "type": e[1]["type"],
                "data": json.loads(e[1]["data"]),
                "timestamp": e[1]["timestamp"]
            }
            for e in events
        ]
    
    def read_by_type(self, event_type: str) -> list:
        all_events = self.read_all()
        return [e for e in all_events if e["type"] == event_type]
    
    def read_by_order(self, order_id: str) -> list:
        all_events = self.read_all()
        return [e for e in all_events if e["data"].get("order_id") == order_id]

event_store = EventStore(r)
print("✅ EventStore ready!")

In [ ]:
print("📜 Event Sourcing Demo")
print("=" * 60)

order_id = str(uuid.uuid4())[:8]

print(f"\n1️⃣ Creating order {order_id}...")
event_store.append("OrderCreated", {
    "order_id": order_id,
    "items": [{"sku": "WIDGET-1", "qty": 2}],
    "total": 99.99
})

print(f"\n2️⃣ Payment charged...")
event_store.append("PaymentCharged", {
    "order_id": order_id,
    "amount": 99.99,
    "transaction_id": "txn_abc123"
})

print(f"\n3️⃣ Inventory reserved...")
event_store.append("InventoryReserved", {
    "order_id": order_id,
    "sku": "WIDGET-1",
    "reservation_id": "res_xyz789"
})

print(f"\n📊 Event history for order {order_id}:")
for event in event_store.read_by_order(order_id):
    print(f"   {event['type']}: {event['data']}")

## 👷 Event-Driven Workers

In [ ]:
class WorkerBase:
    def __init__(self, name: str, event_store: EventStore, redis_client):
        self.name = name
        self.event_store = event_store
        self.redis = redis_client
        self.processed_key = f"worker:{name}:processed"
    
    def was_processed(self, event_id: str) -> bool:
        return self.redis.sismember(self.processed_key, event_id)
    
    def mark_processed(self, event_id: str):
        self.redis.sadd(self.processed_key, event_id)
    
    def process_pending(self):
        events = self.get_relevant_events()
        for event in events:
            if not self.was_processed(event['id']):
                try:
                    self.handle(event)
                    self.mark_processed(event['id'])
                except Exception as e:
                    print(f"      ❌ {self.name} failed: {e}")
    
    def get_relevant_events(self) -> list:
        raise NotImplementedError
    
    def handle(self, event: dict):
        raise NotImplementedError

print("✅ WorkerBase ready!")

In [ ]:
class PaymentWorker(WorkerBase):
    def __init__(self, event_store: EventStore, redis_client):
        super().__init__("payment", event_store, redis_client)
    
    def get_relevant_events(self) -> list:
        return self.event_store.read_by_type("OrderCreated")
    
    def handle(self, event: dict):
        order_id = event['data']['order_id']
        amount = event['data']['total']
        
        print(f"   💳 PaymentWorker: Charging ${amount} for order {order_id}")
        time.sleep(0.3)
        
        if random.random() < 0.2:
            raise Exception("Payment failed")
        
        self.event_store.append("PaymentCharged", {
            "order_id": order_id,
            "amount": amount,
            "transaction_id": f"txn_{uuid.uuid4().hex[:8]}"
        })
        print(f"      ✅ Payment successful!")

class InventoryWorker(WorkerBase):
    def __init__(self, event_store: EventStore, redis_client):
        super().__init__("inventory", event_store, redis_client)
    
    def get_relevant_events(self) -> list:
        return self.event_store.read_by_type("PaymentCharged")
    
    def handle(self, event: dict):
        order_id = event['data']['order_id']
        
        print(f"   📦 InventoryWorker: Reserving for order {order_id}")
        time.sleep(0.2)
        
        self.event_store.append("InventoryReserved", {
            "order_id": order_id,
            "reservation_id": f"res_{uuid.uuid4().hex[:8]}"
        })
        print(f"      ✅ Inventory reserved!")

class ShippingWorker(WorkerBase):
    def __init__(self, event_store: EventStore, redis_client):
        super().__init__("shipping", event_store, redis_client)
    
    def get_relevant_events(self) -> list:
        return self.event_store.read_by_type("InventoryReserved")
    
    def handle(self, event: dict):
        order_id = event['data']['order_id']
        
        print(f"   🚚 ShippingWorker: Creating label for order {order_id}")
        time.sleep(0.2)
        
        self.event_store.append("OrderShipped", {
            "order_id": order_id,
            "tracking_number": f"1Z{uuid.uuid4().hex[:12].upper()}"
        })
        print(f"      ✅ Shipping label created!")

payment_worker = PaymentWorker(event_store, r)
inventory_worker = InventoryWorker(event_store, r)
shipping_worker = ShippingWorker(event_store, r)

print("✅ Workers ready!")

In [ ]:
print("👷 Event-Driven Workflow Demo")
print("=" * 60)

r.flushall()

new_order_id = str(uuid.uuid4())[:8]
print(f"\n1️⃣ New order arrives: {new_order_id}")
event_store.append("OrderCreated", {
    "order_id": new_order_id,
    "items": [{"sku": "GADGET-X", "qty": 1}],
    "total": 149.99
})

print("\n2️⃣ Workers process events (like in production)...")
for _ in range(3):
    payment_worker.process_pending()
    inventory_worker.process_pending()
    shipping_worker.process_pending()
    time.sleep(0.1)

print(f"\n📊 Complete event history:")
for event in event_store.read_all():
    print(f"   {event['type']}")

## 💥 Crash Recovery with Events

In [ ]:
print("💥 Crash Recovery Demo")
print("=" * 60)

r.flushall()

crash_order_id = str(uuid.uuid4())[:8]
print(f"\n1️⃣ Order created: {crash_order_id}")
event_store.append("OrderCreated", {
    "order_id": crash_order_id,
    "items": [{"sku": "THING-1", "qty": 1}],
    "total": 50.00
})

print("\n2️⃣ Payment worker processes, then server CRASHES! 💥")
payment_worker.process_pending()
print("   ... CRASH! All in-memory state lost ...")

print("\n3️⃣ Server restarts with NEW worker instances...")
new_payment_worker = PaymentWorker(event_store, r)
new_inventory_worker = InventoryWorker(event_store, r)
new_shipping_worker = ShippingWorker(event_store, r)

print("\n4️⃣ Workers check events and continue...")
new_payment_worker.process_pending()
new_inventory_worker.process_pending()
new_shipping_worker.process_pending()

print(f"\n📊 Events show complete history:")
for event in event_store.read_by_order(crash_order_id):
    print(f"   ✅ {event['type']}")

print("\n✅ Recovered from crash without losing progress!")

## 😰 Problems with Event Sourcing

In [ ]:
print("😰 Event Sourcing Challenges")
print("=" * 60)
print("""
We've built a decent system, BUT:

1. WORKFLOW IS IMPLICIT
   ─────────────────────────────────────────────────────────
   The workflow is hidden in worker subscriptions.
   - PaymentWorker listens to OrderCreated
   - InventoryWorker listens to PaymentCharged
   - etc.
   
   Hard to see the full flow at a glance!

2. DEBUGGING IS HARD
   ─────────────────────────────────────────────────────────
   Why did this order fail?
   - Which event triggered which worker?
   - What was the order of events?
   - Why is there no ShippingCreated event?

3. COMPENSATION IS MESSY
   ─────────────────────────────────────────────────────────
   If inventory fails, who triggers refund?
   - Need RefundWorker listening to InventoryFailed
   - What if refund fails?
   - Compensation chains get complex

4. NO BUILT-IN TIMEOUTS
   ─────────────────────────────────────────────────────────
   What if payment webhook never comes?
   - Need separate timeout monitoring
   - Manual alerting
   - Not part of workflow definition

5. VERSIONING IS PAINFUL
   ─────────────────────────────────────────────────────────
   Want to add a new step?
   - Old events don't have new fields
   - Running workflows break
   - Migration headaches
""")

## 🧪 Quick Quiz

1. **How does event sourcing help with crash recovery?**

2. **Why is the workflow "implicit" in event sourcing?**

3. **What's still missing that we need?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Crash recovery with events:")
print("   - Events are persisted to durable log")
print("   - Workers track what they've processed")
print("   - After crash, read events and continue")
print()
print("2. Why workflow is implicit:")
print("   - No single workflow definition")
print("   - Flow defined by worker subscriptions")
print("   - Must trace through multiple workers")
print()
print("3. What's still missing:")
print("   - Explicit workflow definition")
print("   - Built-in timeouts and timers")
print("   - Easy compensation handling")
print("   - Workflow versioning")

## 📚 Summary

### What Event Sourcing Provides

1. **Full audit trail** - Every event recorded
2. **Crash recovery** - Replay from events
3. **Decoupling** - Workers are independent
4. **Scalability** - Add more workers

### What's Still Hard

1. Workflow definition is scattered
2. Debugging requires tracing events
3. Compensation logic is complex
4. No built-in timers or signals

### Next Up

In **Notebook 4**, we introduce Temporal:
- Write workflows as code
- Automatic state management
- Built-in retry and timeout